# Appendix Full Results Table Generation
This notebook is for generating tables which show overall model performance across all experiments

## Setup

In [2]:
import matplotlib.pyplot as plt
import numpy as np 
import pandas as pd

In [3]:
# Move up to main repo directory
import os
os.chdir("../..") # ONLY RUN THIS ONCE

In [4]:
## Load CSV file from file path string

def load_csv(file_path):
    """Load a CSV file and return a pandas DataFrame."""
    try:
        df = pd.read_csv(file_path)
        print(f"Loaded CSV file: {file_path}")
        return df
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        return None

In [6]:
rewards_per_scale = {
    'high_scale': [75,50,25],
    'low_scale': [0.75,0.5,0.25],
    'high_neg_scale': [-25, -50, -75],
    'low_neg_scale': [-0.25, -0.5, -0.75]
}

## Main Logic

In [5]:
## First, we load in the merged dataframe
eval_path = 'experiments/merged_eval_results.csv'
results_df = load_csv(eval_path)

# metrics = ['cum_regret', 'sem_optimal_actions', 'reward_optimal_actions']
metrics = ['cum_regret']

Loaded CSV file: experiments/merged_eval_results.csv


In [7]:
results_df

,Unnamed: 0,subfolder,n_shuffles,nomenclature,scale,cum_reward_0_mean,cum_reward_0_std,step_reward_0_mean,step_reward_0_std,step_regret_0_mean,...,sem_optimal_actions_9_mean,sem_optimal_actions_9_std,greedy_probs_9_mean,greedy_probs_9_std,exploration_count_9_mean,exploration_count_9_std,Model,History,Domain,Variance
0,0,alphanumeric_high_neg_scale,10,alphanumeric,high_neg_scale,-50.000000,19.364917,-50.000000,19.364917,25.000000,...,NaN,NaN,0.6,0.489898,3.0,0.0,Qwen8B,Full,Farm,No
1,1,alphanumeric_high_scale,10,alphanumeric,high_scale,52.500000,17.500000,52.500000,17.500000,22.500000,...,NaN,NaN,0.8,0.400000,3.0,0.0,Qwen8B,Full,Farm,No
2,2,alphanumeric_low_neg_scale,10,alphanumeric,low_neg_scale,-0.525000,0.207666,-0.525000,0.207666,0.275000,...,NaN,NaN,0.6,0.489898,3.0,0.0,Qwen8B,Full,Farm,No
3,3,alphanumeric_low_scale,10,alphanumeric,low_scale,0.475000,0.235850,0.475000,0.235850,0.275000,...,NaN,NaN,0.7,0.458258,3.0,0.0,Qwen8B,Full,Farm,No
4,4,sem_rel_helpful_high_neg_scale,10,sem_rel_helpful,high_neg_scale,-25.000000,0.000000,-25.000000,0.000000,0.000000,...,0.5,0.5,0.5,0.500000,3.0,0.0,Qwen8B,Full,Farm,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
294,294,sent_helpful_low_scale,10,sent_helpful,low_scale,0.565493,0.204948,0.565493,0.204948,0.184507,...,1.0,0.0,1.0,0.000000,3.0,0.0,Qwen14B,Summarized,Bandit,High
295,295,sent_mislead_high_neg_scale,10,sent_mislead,high_neg_scale,-36.132757,24.369563,-36.132757,24.369563,11.132757,...,1.0,0.0,1.0,0.000000,3.0,0.0,Qwen14B,Summarized,Bandit,High
296,296,sent_mislead_high_scale,10,sent_mislead,high_scale,50.991114,15.819023,50.991114,15.819023,24.008886,...,1.0,0.0,1.0,0.000000,3.0,0.0,Qwen14B,Summarized,Bandit,High
297,297,sent_mislead_low_neg_scale,10,sent_mislead,low_neg_scale,-0.377592,0.201616,-0.377592,0.201616,0.127592,...,1.0,0.0,1.0,0.000000,3.0,0.0,Qwen14B,Summarized,Bandit,High


In [29]:
from itertools import product

# Standardize string columns to avoid matching issues
for col in ['Model', 'History', 'Variance', 'Domain', 'scale', 'nomenclature']:
    results_df[col] = results_df[col].astype(str).str.strip()

scale_order = ['high_scale', 'low_scale', 'low_neg_scale', 'high_neg_scale']

models = results_df['Model'].unique()
histories = results_df['History'].unique()
variances = results_df['Variance'].unique()
nomenclatures = results_df['nomenclature'].unique()

def latex_escape(s):
    """Escape LaTeX special characters in a string."""
    if not isinstance(s, str):
        s = str(s)
    s = s.replace('_', r'\_')
    # Add more replacements as needed
    return s

for model, history, variance in product(models, histories, variances):
    subset = results_df[
        (results_df['Model'] == model) &
        (results_df['History'] == history) &
        (results_df['Variance'] == variance)
    ]
    if subset.empty:
        print(f"subset {model} {history} {variance} empty!")
        continue

    subset = subset.copy()
    subset['scale'] = pd.Categorical(subset['scale'], categories=scale_order, ordered=True)
    subset = subset.sort_values(['Domain', 'scale'])

    # Get unique (Domain, scale) pairs in order
    domain_scale_pairs = subset[['Domain', 'scale']].drop_duplicates().values.tolist()

    # Use real newlines for LaTeX output and wrap in \resizebox
    header = (
        "\\begin{table}[ht]"
        "\n\\centering"
        "\n\\resizebox{\\textwidth}{!}{%"
        "\n\\renewcommand{\\arraystretch}{1.2}"
        "\n\\begin{tabular}{|l|l|" + "|".join(["ccc" for _ in nomenclatures]) + "|}"
        "\n\\hline"
        "\n\\textbf{Domain} & \\textbf{Scale} "
    )
    for nom in nomenclatures:
        header += f"& \\multicolumn{{3}}{{c|}}{{\\textbf{{{latex_escape(nom)}}}}} "
    header += "\\\\\n\\hline"
    header += "\n & & " + " & ".join(["regret@1 & regret@10 & Cum."] * len(nomenclatures)) + " \\\\"
    header += "\n\\hline"

    rows = ""
    last_domain = None
    for domain, scale in domain_scale_pairs:
        if last_domain is not None and domain != last_domain:
            rows += "\\hline\n"
        row_str = f"{latex_escape(domain)} & {latex_escape(scale)}"
        for nom in nomenclatures:
            nom_row = subset[
                (subset['Domain'] == domain) &
                (subset['scale'] == scale) &
                (subset['nomenclature'] == nom)
            ]
            if not nom_row.empty:
                nom_row = nom_row.iloc[0]
                row_str += f" & {nom_row['step_regret_0_mean']:.2f} & {nom_row['step_regret_9_mean']:.2f} & {nom_row['cum_regret_9_mean']:.2f}"
            else:
                row_str += " & -- & -- & --"
        row_str += " \\\\"
        rows += row_str + "\n"
        last_domain = domain
    rows += "\\hline\n"

    caption = f"Mean regrets for Model={latex_escape(model)}, History={latex_escape(history)}, Variance={latex_escape(variance)}"
    label = f"tab:{latex_escape(model)}_{latex_escape(history)}_{latex_escape(variance)}".replace(" ", "_")

    table = (
        header +
        "\n" +
        rows +
        "\\end{tabular}}\n"  # close tabular and resizebox
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        "\\end{table}"
    )
    print(table)

    # Save to file
    save_path = f"./tables/regret/{history}/{model}/{variance}/table.txt"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w") as f:
        f.write(table)


\begin{table}[ht]
\centering
\resizebox{\textwidth}{!}{%
\renewcommand{\arraystretch}{1.2}
\begin{tabular}{|l|l|ccc|ccc|ccc|ccc|ccc|}
\hline
\textbf{Domain} & \textbf{Scale} & \multicolumn{3}{c|}{\textbf{alphanumeric}} & \multicolumn{3}{c|}{\textbf{sem\_rel\_helpful}} & \multicolumn{3}{c|}{\textbf{sem\_rel\_mislead}} & \multicolumn{3}{c|}{\textbf{sent\_helpful}} & \multicolumn{3}{c|}{\textbf{sent\_mislead}} \\
\hline
 & & regret@1 & regret@10 & Cum. & regret@1 & regret@10 & Cum. & regret@1 & regret@10 & Cum. & regret@1 & regret@10 & Cum. & regret@1 & regret@10 & Cum. \\
\hline
Bandit & high\_scale & 31.25 & 0.00 & 106.25 & 0.00 & 0.00 & 3.12 & 50.00 & 50.00 & 500.00 & 12.50 & 6.25 & 115.62 & 28.12 & 12.50 & 215.62 \\
Bandit & low\_scale & 0.19 & 0.06 & 1.44 & 0.00 & 0.00 & 0.31 & 0.50 & 0.34 & 3.22 & 0.03 & 0.03 & 0.88 & 0.34 & 0.06 & 1.25 \\
Bandit & low\_neg\_scale & 0.22 & 0.06 & 1.44 & 0.00 & 0.07 & 1.21 & 0.50 & 0.06 & 1.94 & 0.09 & 0.16 & 1.72 & 0.38 & 0.03 & 1.19 \\
Bandit & hig